# Knowledge Base Builder — AI Nutritionist RAG
Fetches USDA nutrient data automatically, extracts + chunks + embeds any guideline PDFs you upload (ADA/AHA/WHO etc.), and writes `knowledge_vectors.json` in the exact schema the Android `VectorStore.kt` expects.

**Steps:** run cells 1→2→3 in order. Upload PDFs when prompted in cell 3 (optional — skip if you only want USDA data for now).

## 1. Install dependencies

In [ ]:
!pip install -q sentence-transformers pypdf requests


## 2. USDA FoodData Central — automated fetch
Free API, no scraping needed. Get a free key at https://fdc.nal.usda.gov/api-key-signup — DEMO_KEY works for light testing but is rate-limited.

In [ ]:
import requests

USDA_API_KEY = "l3glk9WKvUvfVaPG4Og5gxeE7B53PkPOmA0HxKZH"
def fetch_usda_foods(query, page_size=10):
    url = "https://api.nal.usda.gov/fdc/v1/foods/search"
    params = {"api_key": USDA_API_KEY, "query": query, "pageSize": page_size}
    r = requests.get(url, params=params)
    r.raise_for_status()
    return r.json().get("foods", [])

def usda_food_to_chunk(food):
    name = food.get("description", "Unknown food")
    nutrients = food.get("foodNutrients", [])
    nutrient_lines = []
    for n in nutrients[:8]:
        nm = n.get("nutrientName")
        val = n.get("value")
        unit = n.get("unitName")
        if nm and val is not None:
            nutrient_lines.append(f"{nm}: {val}{unit}")
    text = f"{name} — " + "; ".join(nutrient_lines)
    return {
        "id": f"usda-{food.get('fdcId')}",
        "text": text,
        "source": "USDA FoodData Central"
    }

# Example: pull a handful of food categories relevant to diet planning
queries = ["brown rice", "lentils", "spinach", "chicken breast", "olive oil"]
usda_chunks = []
for q in queries:
    try:
        foods = fetch_usda_foods(q, page_size=3)
        usda_chunks.extend(usda_food_to_chunk(f) for f in foods)
    except Exception as e:
        print(f"Skipped '{q}': {e}")

print(f"Fetched {len(usda_chunks)} USDA chunks")
usda_chunks[:2]


## 3. PDF guidelines — upload, extract, chunk
Upload any guideline PDFs you've manually downloaded (WHO / ADA / AHA / local research). This step is manual-source, automated-processing: you choose which PDFs are trustworthy, the code does extraction + chunking.

In [ ]:
from google.colab import files
from pypdf import PdfReader
import re

uploaded = files.upload()  # choose one or more PDFs; skip if none

def extract_text(pdf_path):
    reader = PdfReader(pdf_path)
    return "\n".join(page.extract_text() or "" for page in reader.pages)

def chunk_text(text, source_name, chunk_words=180, overlap=30):
    words = re.sub(r"\s+", " ", text).split(" ")
    chunks = []
    i = 0
    idx = 0
    while i < len(words):
        piece = " ".join(words[i:i + chunk_words]).strip()
        if len(piece) > 40:  # skip near-empty fragments
            chunks.append({
                "id": f"{source_name.lower().replace(' ', '-')}-{idx}",
                "text": piece,
                "source": source_name
            })
            idx += 1
        i += chunk_words - overlap
    return chunks

pdf_chunks = []
for fname in uploaded.keys():
    # Set a readable source label per file — edit this if the filename isn't descriptive
    source_label = fname.rsplit(".", 1)[0].replace("_", " ").title()
    text = extract_text(fname)
    file_chunks = chunk_text(text, source_label)
    pdf_chunks.extend(file_chunks)
    print(f"{fname}: {len(file_chunks)} chunks")

print(f"Total PDF chunks: {len(pdf_chunks)}")


Saving standards-of-care-2026.pdf to standards-of-care-2026.pdf
Saving BGD214404.pdf to BGD214404.pdf
Saving Diabetes_Care_BADAS_guideline2019.pdf to Diabetes_Care_BADAS_guideline2019.pdf
Saving Dietary_Guidelines_for_Americans_2020-2025.pdf to Dietary_Guidelines_for_Americans_2020-2025.pdf
Saving lichtenstein-et-al-2026-2026-dietary-guidance-to-improve-cardiovascular-health-a-scientific-statement-from-the-american.pdf to lichtenstein-et-al-2026-2026-dietary-guidance-to-improve-cardiovascular-health-a-scientific-statement-from-the-american (1).pdf
standards-of-care-2026.pdf: 2485 chunks
BGD214404.pdf: 692 chunks
Diabetes_Care_BADAS_guideline2019.pdf: 0 chunks
Dietary_Guidelines_for_Americans_2020-2025.pdf: 421 chunks
lichtenstein-et-al-2026-2026-dietary-guidance-to-improve-cardiovascular-health-a-scientific-statement-from-the-american (1).pdf: 68 chunks
Total PDF chunks: 3666


## 4. Embed everything
Uses `all-MiniLM-L6-v2` (384-dim, small, fast — good default for on-device retrieval). **Important:** whatever embedding model you use here must match the embedding model used on-device in `Embedder.embed()`, or similarity search will be meaningless. If the Android dev picks a different on-device embedder, swap the model name below to match.

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

all_chunks = usda_chunks + pdf_chunks
texts = [c["text"] for c in all_chunks]
embeddings = model.encode(texts, show_progress_bar=True, normalize_embeddings=True)

for c, emb in zip(all_chunks, embeddings):
    c["embedding"] = emb.tolist()

print(f"Embedded {len(all_chunks)} chunks, dim={len(embeddings[0]) if len(embeddings) else 0}")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/116 [00:00<?, ?it/s]

Embedded 3681 chunks, dim=384


## 5. Write `knowledge_vectors.json` in the schema `VectorStore.kt` expects

In [ ]:
import json

output = {"chunks": all_chunks}

with open("knowledge_vectors.json", "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print(f"Wrote knowledge_vectors.json with {len(all_chunks)} chunks")

from google.colab import files as colab_files
colab_files.download("knowledge_vectors.json")


Wrote knowledge_vectors.json with 3681 chunks


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Next steps
- Drop the downloaded `knowledge_vectors.json` into `app/src/main/assets/` in the Android project, replacing the placeholder file
- Confirm the on-device `Embedder` implementation uses the **same** model (`all-MiniLM-L6-v2` or whichever you swapped in) — mismatch breaks retrieval
- Re-run this notebook any time you add more source PDFs or want more USDA food categories